In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data= pd.read_csv("../datasets/dataset_train.csv")
pred = pd.read_csv("../prediction.csv")

target="Hogwarts House"

labels = data[target].unique().tolist()

# X = train[""]

fig, ax = plt.subplots()

sns.countplot(x=target, data=data, alpha=0.4,ax=ax)
sns.countplot(x=target, data=pred, alpha=0.4, color="orange",ax=ax)

# plt.show()

In [ ]:
cols_to_drop = ["Index", "First Name", "Last Name", "Birthday"]
data = data[data.columns.difference(cols_to_drop).tolist()]  # type: ignore

categorical_features = ["Best Hand"]
num_features = data.columns.difference(categorical_features).tolist() # type: ignore

fig, axes = plt.subplots(5,2, figsize=(16,18))
for ax, col in zip(axes.flat, num_features):
    sns.kdeplot(data=data[col],ax=ax) # type: ignore


fig.show()

In [ ]:
from sklearn.model_selection import train_test_split
from logreg_predict import predict
import joblib

X = data.drop(columns=target)
y = data[target]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.2)



In [ ]:
# X_train["Best Hand"] = X_train["Best Hand"].map({"Left": 0, "Right": 1})  # type: ignore

# drop noise + target
cols_to_drop = ["Index", "First Name", "Last Name", "Birthday"]
X_train = X_train[X_train.columns.difference(cols_to_drop).tolist()]  # type: ignore

categorical_features = ["Best Hand"]
num_features = X_train.columns.difference(categorical_features).tolist()

# # handle na
# X_train[num_features] = X_train[num_features].fillna(X_train[num_features].median())  # type: ignore
# X_train[categorical_features] = X_train[categorical_features].fillna(X_train[categorical_features].mode().iloc[0])  # type: ignore

# X_valid[num_features] = X_valid[num_features].fillna(X_valid[num_features].median())  # type: ignore
# X_valid[categorical_features] = X_valid[categorical_features].fillna(X_valid[categorical_features].mode().iloc[0])  # type: ignore

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

num_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler())
    ])

cat_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder())
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, categorical_features)
], remainder="passthrough")

import numpy as np

X_train_proc: np.ndarray = preprocessor.fit_transform(X_train)
X_valid_proc  = preprocessor.transform(X_valid)

dff = pd.DataFrame(X_valid_proc, columns=preprocessor.get_feature_names_out())
dff


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_valid_enc = le.transform(y_valid)

labels=le.classes_.tolist()


In [ ]:
# from math import exp
import importlib
import ft_logistic_regression, config
importlib.reload(ft_logistic_regression)
importlib.reload(config)
from ft_logistic_regression import FtLogisticRegression
from config import MODEL_PATH

ft_lr = FtLogisticRegression()

ft_lr.fit(X_train_proc, y_train_enc)

joblib.dump(ft_lr, MODEL_PATH)

In [ ]:


lr = joblib.load(MODEL_PATH)

ft_p = lr.predict(X_valid_proc)
print(ft_p.shape)
ft_p

lr.loss_

plt.plot(lr.loss_)


In [ ]:
from sklearn.linear_model import LogisticRegression


lr = LogisticRegression()

lr.fit(X_train_proc, y_train_enc)

p = lr.predict(X_valid_proc)
print(p.shape)
p



In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
)
from sklearn.utils.multiclass import unique_labels

def evaluate(name, model, X_test, y_test):
    pred = model.predict(X_valid_proc)
    proba = model.predict_proba(X_valid_proc)
    cm = confusion_matrix(y_test, pred, normalize="true")
    cm_plot = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    # cm_plot.plot()

    cr = classification_report(y_test, pred, target_names=labels)

    auc = roc_auc_score(y_test, proba, multi_class="ovr")
    precision = precision_score(y_test, pred, average="macro")

    print("AUC score: ", auc)
    print("Precision score: ", precision)
    print(cr)

evaluate("Log reg", lr, X_valid_proc, y_valid_enc)


In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import label_binarize

class_indices = le.classes_

proba = lr.predict_proba(X_valid_proc)
y_valid_bin = label_binarize(y_valid_enc, classes=range(len(class_indices)))

score = average_precision_score(y_valid_bin, proba, average="macro")


precision, recall, _ = precision_recall_curve(y_valid_bin.ravel(), proba.ravel())
plt.plot(recall, precision)

plt.xlabel("recall")
plt.ylabel("precision")
plt.title("PR curve")






In [ ]:
# explainability

import shap

features = preprocessor.get_feature_names_out()
features = [f.strip("num__") for f in features]


X_train_df = pd.DataFrame(X_train_proc, columns=features)
X_valid_df = pd.DataFrame(X_valid_proc, columns=features)



masker = shap.maskers.Independent(X_train_df, max_samples=len(X_train_df))

explainer = shap.LinearExplainer(lr, masker=masker)



shap_values = explainer.shap_values(X_valid_df)

fig, axes = plt.subplots(len(class_indices), figsize=(20, 6))

for i, cl in enumerate(class_indices):
    plt.sca(axes[i])
    shap.summary_plot(shap_values[:,:,i], X_valid_df, title=cl, show=False, plot_size=(10,30))
    plt.title(cl)

plt.show()